# Automatización LCK A320F (NB) — desde BigQuery hasta el archivo de bloques

Este notebook cubre, de punta a punta, **la Fase 0 (Insumos) y la Fase 3 (Búsqueda de vuelos)**
del "Procedimiento_LCK_A320F_Flujograma_y_Paso_a_Paso.docx" (pasos 14 a 20 del flujograma),
para la flota NB (A320/A319).

**Todo el código de las celdas de filtrado/armado es una copia literal** (no una reescritura)
de `generar_candidatos_nb.py` y `generar_reporte_pairings_nb.py`, ya validados con QA
(recálculo independiente en Python, no solo revisión estructural) en varias corridas
anteriores contra datos reales. Solo cambia la fuente: en vez de leer un CSV/JSON local,
se consulta BigQuery directamente.

**Qué NO hace este notebook** (falta la Fase 1, 2, 4 y 5 completas — ver la celda final
"Estado y próximos pasos" para el detalle exacto de qué falta y qué preguntar a otras personas).


## 1. Instalar dependencias y autenticar

In [ ]:
# Colab ya trae pandas/google-cloud-bigquery; openpyxl normalmente no.
!pip install -q openpyxl


In [ ]:
import sys
import re
import pandas as pd
from google.colab import auth
from google.cloud import bigquery
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.comments import Comment
from openpyxl.worksheet.datavalidation import DataValidation

auth.authenticate_user()
print("Autenticado con tu cuenta de Google.")


In [ ]:
# Ajusta el proyecto de billing si usas uno distinto para correr la query.
PROJECT_ID = "operations-data-prod"
client = bigquery.Client(project=PROJECT_ID)


## 2. Query a BigQuery (Fase 0, pasos 01-05)

Reemplaza la descarga manual del Panel (Looker Studio) + filtros FP/SAB + recorte de columnas.

**Corrección aplicada:** el archivo `automatizacion_bigquery/consulta BQ NB.sql` guardado en el
repo tiene `subfleet_code IN ('319', '319')` (typo, se repite 319 dos veces) — acá está
corregido a `IN ('319', '320')`. Si vuelves a copiar la query desde ese `.sql`, corrige ese
typo también ahí, o vas a perder toda la flota A320 en la próxima corrida.


In [ ]:
QUERY_NB = """
SELECT
  pairing_id                       AS trip,
  flight_start_date_local_time     AS inicio_vuelo_lt,
  duty_calendar_day_number         AS dia_duty,
  flight_number                    AS vuelo,
  departure_airport_code           AS dep,
  arrival_airport_code             AS arr,
  flight_departure_time_crew_base  AS std_hb,
  flight_arrival_hour_block_time   AS sta_hb,
  flight_block_time                AS hbt,
  subfleet_code                    AS sub_fleet

FROM `operations-data-prod.carmen_gold.crew_pairing_carmen_system`

WHERE
  flight_start_date_local_time BETWEEN DATE \'2025-09-01\' AND DATE \'2026-09-30\'
  AND subsidiary_code IN (\'LP\')
  AND load_type_code = \'FP\'
  AND crew_range_type_code = \'SAB\'
  AND subfleet_code IN (\'319\', \'320\')          -- NB: narrow body (corregido, ver nota arriba)

QUALIFY
  CASE
    WHEN load_type_code = \'FP\' AND
         DATE(ingestion_datetime) = MAX(CASE WHEN load_type_code = \'FP\' THEN DATE(ingestion_datetime) END)
           OVER (PARTITION BY subsidiary_code, fleet_type_code, crew_range_type_code,
                              reference_month_number, reference_year)
      THEN 0
    WHEN load_type_code = \'ES\' AND
         MAX(CASE WHEN load_type_code = \'FP\' THEN 0 ELSE 0 END)
           OVER (PARTITION BY subsidiary_code, fleet_type_code, crew_range_type_code,
                              reference_month_number, reference_year) = -1 AND
         DATE(ingestion_datetime) = MAX(CASE WHEN load_type_code = \'ES\' THEN DATE(ingestion_datetime) END)
           OVER (PARTITION BY subsidiary_code, fleet_type_code, crew_range_type_code,
                              reference_month_number, reference_year)
      THEN 0
    ELSE -1
  END = 0

ORDER BY pairing_id ASC
"""

df_raw = client.query(QUERY_NB).to_dataframe()
print(f"Filas descargadas: {len(df_raw)}")
df_raw.head()


## 3. Configuración (mes objetivo + reglas 2.9-2.14)

**Fuente de cada regla, para que quede trazable:**
- Rutas válidas: `Procedimiento_LCK_A320F_Flujograma_y_Paso_a_Paso.docx` (reunión Parte 4,
  10-ago-2026), que coincide con el manual original (Parte 2). Reemplazó una lista anterior
  que venía de un video y que incluía IQT / excluía TBP-TCQ — confirmado con el usuario que
  esta es la vigente.
- `EXCLUSIONES_MES`: excepción real declarada en el mismo documento para sept-2026 (AQP fuera
  por el Perumín). **Revisar cada mes si sigue vigente** — no asumir que se repite.
- El resto (horarios, conexión, PSV, HBT) sale directo del manual 2.12 y fue confirmado por
  Fernando con ejemplos reales (ver `feedback_qa_rigor_pairings.md` en `docs/claude-memory`).


In [ ]:
MES_OBJETIVO = 9
ANIO_OBJETIVO = 2026

MESES = {
    "ene": 1, "feb": 2, "mar": 3, "abr": 4, "may": 5, "jun": 6, "jul": 7,
    "ago": 8, "sep": 9, "sept": 9, "oct": 10, "nov": 11, "dic": 12,
}
WD_ES = {
    "Monday": "lunes", "Tuesday": "martes", "Wednesday": "miércoles",
    "Thursday": "jueves", "Friday": "viernes", "Saturday": "sábado",
    "Sunday": "domingo",
}

RUTAS_VALIDAS_ARR = {"AQP", "CIX", "CJA", "CUZ", "PEM", "PIU", "TBP", "TPP", "TCQ"}

# Exclusiones dinámicas del mes -> revisar/actualizar cada corrida.
EXCLUSIONES_MES = {"AQP"}

HORA_MIN_SALIDA = pd.Timedelta(hours=8, minutes=30)   # 2.12: sale después de 08:30
HBT_MIN = pd.Timedelta(hours=1)                        # 2.12: HBT > 1 hora
CONEXION_MIN = pd.Timedelta(minutes=50)                # 2.12: conexión >= 50 min
CONEXION_MAX = pd.Timedelta(hours=1, minutes=30)       # 2.12: conexión < 1h30
PSV_MAX = pd.Timedelta(hours=11)                       # 2.12: PSV <= 11 hrs


## 4. Depurar + corregir reutilización de `pairing_id` (Fase 0, pasos 04-05)

`cargar_bq_a_df` = copia de `cargar_y_depurar` (2.9-2.10 del manual), adaptada para recibir
el DataFrame de BigQuery en vez de leer un CSV.

`separar_instancias_trip` es un bug real encontrado y corregido en esta automatización, que
**no existía con el CSV viejo del Panel**: BigQuery devuelve `pairing_id` reutilizado entre
pairings reales distintos a lo largo de los 13 meses que trae la query (confirmado: 84 de
2689 trips con un rango de fechas de 26-29 días, imposible para un pairing real de pocos
días). Se separa detectando cada vez que `dia_duty` retrocede — nunca debería bajar dentro
de un mismo pairing real.


In [ ]:
def parse_hora(s):
    """'08:15:00' -> Timedelta(hours=8, minutes=15). Sirve con str o datetime.time."""
    if pd.isna(s):
        return pd.NaT
    h, m, sec = str(s).split(":")
    sec = sec.split(".")[0]  # por si BigQuery trae microsegundos
    return pd.Timedelta(hours=int(h), minutes=int(m), seconds=int(sec))


def cargar_bq_a_df(df_raw: pd.DataFrame) -> pd.DataFrame:
    """Copia de cargar_y_depurar (generar_candidatos_nb.py, 2.9-2.10),
    adaptada para recibir el DataFrame de BigQuery en vez de un CSV."""
    df = df_raw.copy()
    df["trip"] = df["trip"].astype(str)
    df["dia_duty"] = df["dia_duty"].astype(int)
    df["fecha_dt"] = pd.to_datetime(df["inicio_vuelo_lt"])
    df["std_td"] = df["std_hb"].apply(parse_hora)
    df["sta_td"] = df["sta_hb"].apply(parse_hora)
    df["hbt_td"] = df["hbt"].apply(parse_hora)
    df["std_dt"] = df["fecha_dt"] + df["std_td"]
    df["sta_dt"] = df["fecha_dt"] + df["sta_td"]
    # si la hora de llegada es menor que la de salida, cruzó medianoche
    df.loc[df["sta_dt"] < df["std_dt"], "sta_dt"] += pd.Timedelta(days=1)
    df["dia_semana"] = df["fecha_dt"].dt.day_name().map(WD_ES)
    return df


def separar_instancias_trip(df: pd.DataFrame) -> pd.DataFrame:
    """pairing_id se reutiliza entre pairings reales distintos en este
    export. Separa instancias: dentro de un mismo pairing real, dia_duty
    nunca debería retroceder (puede repetirse el mismo día para 2+
    tramos, o saltar por un día de descanso, pero no bajar). Cada vez
    que baja, es un pairing NUEVO reusando el mismo número.

    Agrega "trip_original" (el pairing_id tal cual vino de BigQuery,
    para mostrar) y reemplaza "trip" por una clave sintética única por
    instancia (ej. "2_0", "2_1") -> la que usan las funciones siguientes
    para agrupar, sin que se mezclen dos pairings distintos.
    """
    df = df.sort_values(["trip", "fecha_dt", "std_dt"]).reset_index(drop=True)
    instancia = []
    trip_actual = None
    max_dia_duty = -1
    idx_instancia = 0
    for _, row in df.iterrows():
        if row["trip"] != trip_actual:
            trip_actual = row["trip"]
            idx_instancia = 0
            max_dia_duty = row["dia_duty"]
        elif row["dia_duty"] < max_dia_duty:
            idx_instancia += 1
            max_dia_duty = row["dia_duty"]
        else:
            max_dia_duty = max(max_dia_duty, row["dia_duty"])
        instancia.append(f"{row['trip']}_{idx_instancia}")

    df["trip_original"] = df["trip"]
    df["trip"] = instancia
    return df


## 5. Filtro de mes/ruta y armado de la "primera mitad" (Fase 0 paso 05 + Fase 3 pasos 15-16)

Copia literal de `filtrar_mes_y_ruta` y `armar_primeras_mitades` de `generar_candidatos_nb.py`
(reglas 2.11, 2.12, 2.14 del manual), incluyendo los 3 bugs reales que quedaron corregidos ahí:
1. Ruta triangular con tramo intermedio filtrado (la vuelta no calzaba con la ciudad de la ida).
2. Día 1 "falso": el día que sobrevive el filtro de mes/ruta puede no ser el día 1 real del
   pairing si el día 1 real cayó en otro mes -> se compara contra `dia_duty_min_real`
   (calculado ANTES de filtrar).
3. Día 1 con 6, 8 o 10 tramos (3-5 idas+vueltas seguidas): se descarta el trip completo, no
   solo se toman los primeros 2 tramos (indicación explícita de Fernando).


In [ ]:
def filtrar_mes_y_ruta(df: pd.DataFrame, mes: int, anio: int) -> pd.DataFrame:
    """2.11: mes objetivo, dep=LIM, ruta nacional válida."""
    rutas_validas = RUTAS_VALIDAS_ARR - {c.upper() for c in EXCLUSIONES_MES}

    en_mes = (df["fecha_dt"].dt.month == mes) & (df["fecha_dt"].dt.year == anio)
    sale_de_lim = df["dep"] == "LIM"
    llega_a_lim = df["arr"] == "LIM"
    ruta_nacional_ok = df["arr"].isin(rutas_validas) | (llega_a_lim)

    return df[en_mes & (sale_de_lim | llega_a_lim) & ruta_nacional_ok].copy()


def armar_primeras_mitades(df: pd.DataFrame, dia_duty_min_real=None):
    """
    2.14: para cada trip, la "primera mitad" son las piernas del
    dia_duty mínimo (el instructor solo cubre el primer día de duty).

    IMPORTANTE: `df` ya viene filtrado por mes/ruta, así que su dia_duty
    mínimo puede NO ser el día 1 real del pairing si ese día 1 cayó en
    otro mes o en una ruta no válida (y por eso se filtró). Para no
    chequear crew en un día que en realidad es el 2do o 3ro del pairing
    (violaría 2.14), se recibe `dia_duty_min_real`: el dia_duty mínimo
    de cada trip calculado ANTES de filtrar.

    Devuelve (candidatos_validos, excluidos_con_motivo).
    """
    validos_rows = []
    excluidos_rows = []

    for trip, grupo in df.groupby("trip"):
        dia_min = grupo["dia_duty"].min()
        if dia_duty_min_real is not None and trip in dia_duty_min_real.index \
                and dia_min != dia_duty_min_real.loc[trip]:
            excluidos_rows.append({
                "trip": trip,
                "motivo": (f"el día {dia_min} que sobrevivió el filtro no es el día 1 real del "
                           f"pairing (día 1 real = {dia_duty_min_real.loc[trip]}, cayó fuera de "
                           f"mes/ruta) -> el instructor solo puede cubrir el día 1 real"),
            })
            continue
        dia1 = grupo[grupo["dia_duty"] == dia_min].sort_values("std_dt")

        if len(dia1) < 2:
            excluidos_rows.append({"trip": trip, "motivo": "día 1 sin vuelta el mismo día (1 solo tramo)"})
            continue

        # Indicación de Fernando: si el día 1 trae 6, 8 o 10 tramos (3, 4
        # o 5 idas+vueltas seguidas), es demasiado para un solo instructor
        # -> se descarta el trip completo (no solo los primeros 2 tramos).
        if len(dia1) in (6, 8, 10):
            excluidos_rows.append({"trip": trip, "motivo": f"día 1 tiene {len(dia1)} tramos -> no se toma"})
            continue

        ida, vuelta = dia1.iloc[0], dia1.iloc[1]
        if ida["dep"] != "LIM":
            excluidos_rows.append({"trip": trip, "motivo": "el primer tramo no sale de LIM"})
            continue
        if vuelta["arr"] != "LIM":
            excluidos_rows.append({"trip": trip, "motivo": "el segundo tramo del día 1 no vuelve a LIM"})
            continue
        if vuelta["dep"] != ida["arr"]:
            excluidos_rows.append({"trip": trip, "motivo": f"vuelta sale de {vuelta['dep']} pero la ida llegó a {ida['arr']} (ruta triangular con tramo intermedio filtrado)"})
            continue

        motivos = []
        if (ida["std_dt"] - ida["fecha_dt"]) <= HORA_MIN_SALIDA:
            motivos.append("sale antes/igual a 08:30")
        if ida["hbt_td"] <= HBT_MIN:
            motivos.append("HBT ida <= 1h")
        if vuelta["hbt_td"] <= HBT_MIN:
            motivos.append("HBT vuelta <= 1h")

        # La conexión >=50min y <1h30 (2.12) es para EMPATAR DOS PAIRINGS
        # DISTINTOS el mismo día (paso 17 del flujograma), NO la vuelta
        # interna de un mismo pairing -> confirmado con datos: el 95% de
        # las conexiones internas reales están entre 35 y 50 min. Por eso
        # NO se usa como motivo de exclusión acá, solo queda informativa.
        conexion = vuelta["std_dt"] - ida["sta_dt"]
        if conexion <= pd.Timedelta(0):
            motivos.append(f"conexión interna negativa/cero ({conexion}) -> datos inconsistentes")

        psv = vuelta["sta_dt"] - ida["std_dt"]
        if psv > PSV_MAX:
            motivos.append(f"PSV {psv} > 11h")

        if motivos:
            excluidos_rows.append({"trip": trip, "motivo": "; ".join(motivos)})
            continue

        validos_rows.append({
            "Fecha": ida["fecha_dt"].strftime("%d/%m/%Y"),
            "DíaSem": ida["dia_semana"],
            "Pairing ID": trip,
            "Vuelo Ida": ida["vuelo"], "Dep": ida["dep"], "Arr": ida["arr"],
            "STD Ida": ida["std_hb"], "STA Ida": ida["sta_hb"], "HBT Ida": ida["hbt"],
            "Vuelo Vuelta": vuelta["vuelo"], "Dep Vta": vuelta["dep"], "Arr Vta": vuelta["arr"],
            "STD Vuelta": vuelta["std_hb"], "STA Vuelta": vuelta["sta_hb"], "HBT Vuelta": vuelta["hbt"],
            "Conexión": str(conexion), "PSV Total": str(psv),
            "Sub Flota": ida["sub_fleet"],
            "_orden": ida["std_dt"],
            "_ida_std_dt": ida["std_dt"], "_vuelta_sta_dt": vuelta["sta_dt"],
        })

    cols_finales = ["Fecha", "DíaSem", "Pairing ID", "Vuelo Ida", "Dep", "Arr", "STD Ida",
                     "STA Ida", "HBT Ida", "Vuelo Vuelta", "Dep Vta", "Arr Vta", "STD Vuelta",
                     "STA Vuelta", "HBT Vuelta", "Conexión", "PSV Total", "Sub Flota",
                     "Posible 2do vuelo (mismo día)"]

    if not validos_rows:
        return pd.DataFrame(columns=cols_finales), pd.DataFrame(excluidos_rows)

    validos = pd.DataFrame(validos_rows).sort_values("_orden")

    # 2.15/paso 17: para cada candidato, qué OTROS candidatos válidos del
    # mismo día podrían ser "el segundo vuelo" del instructor.
    posibles = []
    for _, row in validos.iterrows():
        mismo_dia = validos[validos["Fecha"] == row["Fecha"]]
        ventana_ini = row["_vuelta_sta_dt"] + CONEXION_MIN
        ventana_fin = row["_vuelta_sta_dt"] + CONEXION_MAX
        candidatos_2do = mismo_dia[
            (mismo_dia["Pairing ID"] != row["Pairing ID"]) &
            (mismo_dia["_ida_std_dt"] > ventana_ini) &
            (mismo_dia["_ida_std_dt"] < ventana_fin)
        ]["Pairing ID"].tolist()
        posibles.append(", ".join(str(p) for p in candidatos_2do) if candidatos_2do else "")
    validos["Posible 2do vuelo (mismo día)"] = posibles

    validos = validos[cols_finales].reset_index(drop=True)
    return validos, pd.DataFrame(excluidos_rows)


## 6. Emparejar en bloques de 4 filas = 1 día de instructor (Fase 3, pasos 17-19)

Copia literal de `parear_candidatos` de `generar_reporte_pairings_nb.py`. Incluye el bug real
corregido de PSV combinado (antes solo se validaba el PSV de cada pairing por separado; dos
pairings válidos podían sumar >11h combinados).

In [ ]:
def _fecha_dt(s):
    d, m, a = s.split("/")
    return pd.Timestamp(year=int(a), month=int(m), day=int(d))


def _hora_td(s):
    h, m, sec = s.split(":")
    return pd.Timedelta(hours=int(h), minutes=int(m), seconds=int(sec))


def parear_candidatos(validos: pd.DataFrame):
    """
    Empareja cada candidato con otro del mismo día que conecte en
    (CONEXION_MIN, CONEXION_MAX) Y cuyo PSV COMBINADO (desde la salida
    del primero hasta el regreso del segundo -> el día completo del
    instructor) no supere PSV_MAX. Recorrido cronológico, greedy.

    Devuelve una lista de tuplas (candidato_A, candidato_B_o_None).
    """
    df = validos.copy()
    df["_ida_dt"] = df.apply(lambda r: _fecha_dt(r["Fecha"]) + _hora_td(r["STD Ida"]), axis=1)
    df["_vta_dt"] = df.apply(lambda r: _fecha_dt(r["Fecha"]) + _hora_td(r["STA Vuelta"]), axis=1)
    df = df.sort_values("_ida_dt").reset_index(drop=True)

    usados = set()
    bloques = []
    for i, row in df.iterrows():
        if row["Pairing ID"] in usados:
            continue
        ventana_ini = row["_vta_dt"] + CONEXION_MIN
        ventana_fin = row["_vta_dt"] + CONEXION_MAX
        mismo_dia = df[
            (~df["Pairing ID"].isin(usados)) &
            (df["Pairing ID"] != row["Pairing ID"]) &
            (df["Fecha"] == row["Fecha"]) &
            (df["_ida_dt"] > ventana_ini) & (df["_ida_dt"] < ventana_fin) &
            (df["_vta_dt"] - row["_ida_dt"] <= PSV_MAX)
        ].sort_values("_ida_dt")

        usados.add(row["Pairing ID"])
        if len(mismo_dia) > 0:
            segunda = mismo_dia.iloc[0]
            usados.add(segunda["Pairing ID"])
            bloques.append((row, segunda))
        else:
            bloques.append((row, None))
    return bloques


## 7. Armar el Excel final (Fase 3 paso 20 + Fase 4 paso 21-22 parcial)

Copia literal de `construir_bloques_nb` de `generar_reporte_pairings_nb.py`: hoja "Pairings NB"
(bloques con el grid de reglas idéntico a tu plantilla, dropdown de instructor y de actividad,
conexión resaltada en verde, PSV en amarillo), hoja "Instructores" (catálogo) y hoja
"Resumen Final" (equivalente al CUADRO FINAL del Archivo 10, con CUPOS = 4 si A320, 3 si A319,
según el manual 2.4/2.18 — sin la excepción "Habilitación=2" que no está documentada).

In [ ]:
ACTIVIDADES_NB = [
    "LCK A320F", "Reentrenamiento A320F", "LCK A320F + Habilitación A320F",
    "LCK A320 ALUMNO", "Auditoria CAB Vuelo", "BIANUAL IDE - HAB IDE",
    "EXP RECIENTE A320", "CHEQUEO LATAM A320F", "CHEQUEO DGAC A320F",
]

# Catálogo de instructores (nombre corto, BP, nombre completo/legal) ->
# alimenta la hoja "Instructores" y el desplegable de la columna M.
# Dato dado directamente por Fernando, no derivado de ningún archivo.
INSTRUCTORES_DATA = [
    ("Christian Rondon", "1271571", " RONDON BARRUTIA CHRISTIAN ERIC "),
    ("Erika Davila", "967092", "DAVILA BELLO MARIA ERIKA"),
    ("Sebastian Correa", "2396710", " CORREA GARCIA JUAN SEBASTIAN "),
    ("Fiorella Ruiz", "2713993", "RUIZ RIOJA FIORELLA DEL PILAR"),
    ("Jazmin Guerra", "29530", "GUERRA SUAREZ JAZMIN"),
    ("Jennifert Acurio", "3779550", "ACURIO DARGENT JENNIFERT MILAGROS"),
    ("Karen Santa Cruz", "2843319", "SANTA CRUZ HUAMAN KAREN"),
    ("Luis Bacigalupo", "2963161", "BACIGALUPO FLORES LUIS ENRIQUE"),
    ("Claudia Flores", "3217561", " FLORES FUENTES DAVILA CLAUDIA ALEXANDRA "),
    ("Karla Moz", "71348", "MOZ MONTES KARLA LISSETTE"),
    ("Elizabeth Torres", "2369641", "TORRES POLO ELIZABETH DEL PILAR"),
    ("Patricia Najar", "2369624", "NAJAR CRUZ PATRICIA DEL PILAR"),
    ("Javier Zapata", "3134911", "ZAPATA GARAYAR JAVIER RICARDO SALVADOR"),
    ("Jefferson Mendez", "3750335", " MENDEZ RUCOBA JEFFERSON "),
    ("Gabriela Ungaro", "3852423", "UNGARO GUTIERREZ GABRIELA"),
    ("Mariella Carrasco", "2604360", "CARRASCO BENAVIDES ROSA MARIELLA"),
    ("Cesar Campos", "2823133", " CAMPOS CONCHE CESAR AUGUSTO "),
    ("Kevin Segovia", "3189967", "SEGOVIA TAPIA RAY KEVIN"),
    ("Milagros Salas", "2415373", "SALAS COSIO MILAGROS PATRICIA"),
    ("Judith Fernandez", "2440915", "FERNANDEZ GARCIA JUDITH JULIET"),
    ("Rafael Nieto", "3796947", " NIETO SAENZ RAFAEL ANTONIO "),
    ("Gianfranco Celiz", "3841387", " CELIZ ROSSI GIANFRANCO PAOLO "),
]

# Grid EXACTO de las 4 filas de reglas (fila, columna, texto, es_rojo),
# tal cual tu plantilla -> no es una lista, es una cuadrícula de 4 filas.
TITULOS_GRID = [
    (1, 2, "Conexión > o = a 50min y < a 1hr y 30min", False),
    (1, 4, "NO CONSIDERAR TRU/JUL/JAE/AYP/JAU/IQT", True),
    (1, 8, "CONSIDERAR PAIRINGS PARTIDOS LCK A320", False),
    (1, 11, "Vuelos LCK = Siempre en Flota 320", False),
    (2, 2, "Vuelos HBT mayor a 1 hora", False),
    (2, 4, "PSV NO MAYOR A 11 HRS", False),
    (2, 8, "CONSIDERAR SIEMPRE EL PDR", False),
    (2, 11, "Vuelos iniciando más de 08:30", False),
    (3, 4, "NO REPETIR PAIRING EN LCK", True),
]

HEADERS = ["Pairing ID", "Fecha", "DíaSem", "Vuelo", "Dep", "Arr", "STD", "STA", "Sub Flota"]


In [ ]:
def construir_bloques_nb(bloques: list, out_path: str, n_muestra=None):
    wb = Workbook()
    ws = wb.active
    ws.title = "Pairings NB"

    bold = Font(bold=True)
    red_bold = Font(bold=True, color="CC0000")
    green_bold = Font(bold=True)
    dutyid_fill = PatternFill("solid", fgColor="FFC000")
    psv_fill = PatternFill("solid", fgColor="FFFF00")       # amarillo
    resumen_fill = PatternFill("solid", fgColor="D9E1F2")
    conexion_lim_fill = PatternFill("solid", fgColor="C6E8C6")  # verde pastel
    thin = Side(style="thin", color="999999")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    center = Alignment(horizontal="center", wrap_text=True)

    COL_CONEXION, COL_HBT = 11, 12          # K, L
    COL_INS, COL_VACIA1, COL_DESC = 13, 14, 15   # M, N(vacía), O
    COL_VACIA2 = 16                          # P(vacía)
    COL_RESUMEN_INI = 17                     # Q..X
    RESUMEN_HEADERS = ["Pairing ID", "Fecha", "DíaSem", "Vuelo", "Ruta",
                        "Instructor", "Actividad", "Sub Flota"]

    # --- hoja "Instructores" ---
    wi = wb.create_sheet("Instructores")
    for j, h in enumerate(["Instructor", "BP", "Nombre"]):
        c = wi.cell(row=1, column=1 + j, value=h)
        c.font = bold
        c.border = border
    for i, (nombre, bp, legal) in enumerate(INSTRUCTORES_DATA, start=2):
        wi.cell(row=i, column=1, value=nombre).border = border
        wi.cell(row=i, column=2, value=bp).border = border
        wi.cell(row=i, column=3, value=legal).border = border
    for c, w in zip("ABC", (18, 12, 40)):
        wi.column_dimensions[c].width = w
    n_instructores = len(INSTRUCTORES_DATA)

    dv_instructor = DataValidation(
        type="list", formula1=f"=Instructores!$A$2:$A${1 + n_instructores}", allow_blank=True)
    ws.add_data_validation(dv_instructor)

    # --- grid de reglas (4 filas EXACTAS, no lista) ---
    for fila_r, col_r, texto, es_rojo in TITULOS_GRID:
        c = ws.cell(row=fila_r, column=col_r, value=texto)
        c.font = red_bold if es_rojo else bold

    fila_labels = 6  # deja la fila 4 (parte del grid) y la 5 en blanco

    for j, h in enumerate(RESUMEN_HEADERS):
        c = ws.cell(row=fila_labels, column=COL_RESUMEN_INI + j, value=h)
        c.font = bold
        c.fill = resumen_fill
        c.border = border
        c.alignment = center

    dv = DataValidation(type="list", formula1='"' + ",".join(ACTIVIDADES_NB) + '"', allow_blank=True)
    ws.add_data_validation(dv)

    todas_filas_pairing = []  # (r_ida, r_vta, r1_del_bloque) -> hoja Resumen Final

    HEADER_TITULOS = ["Pairing ID", "FECHA REAL", "Day of Week", "Flight No",
                       "Dep Stn", "Arr Stn", "STD", "STA", "subflota", "Conexion", "HBT"]

    fila = fila_labels + 2  # deja 1 fila en blanco antes del primer bloque
    if n_muestra is not None:
        bloques = bloques[:n_muestra]

    for candA, candB in bloques:
        candidatos_bloque = [candA] + ([candB] if candB is not None else [])
        filas_bloque = []

        for j, titulo in enumerate(HEADER_TITULOS):
            c = ws.cell(row=fila, column=2 + j, value=titulo)
            c.font = bold
            c.fill = resumen_fill
            c.border = border
            c.alignment = center
        fila += 1

        fila_r1 = fila
        for cand in candidatos_bloque:
            r_ida, r_vta = fila, fila + 1
            ida = {"Pairing ID": cand["Pairing ID"], "Fecha": cand["Fecha"], "DíaSem": cand["DíaSem"],
                   "Vuelo": cand["Vuelo Ida"], "Dep": cand["Dep"], "Arr": cand["Arr"],
                   "STD": cand["STD Ida"], "STA": cand["STA Ida"], "Sub Flota": cand["Sub Flota"]}
            vta = {"Pairing ID": cand["Pairing ID"], "Fecha": cand["Fecha"], "DíaSem": cand["DíaSem"],
                   "Vuelo": cand["Vuelo Vuelta"], "Dep": cand["Dep Vta"], "Arr": cand["Arr Vta"],
                   "STD": cand["STD Vuelta"], "STA": cand["STA Vuelta"], "Sub Flota": cand["Sub Flota"]}

            for r, leg in ((r_ida, ida), (r_vta, vta)):
                for j, h in enumerate(HEADERS):
                    c = ws.cell(row=r, column=2 + j, value=leg[h])
                    c.border = border
                    c.alignment = center
                c = ws.cell(row=r, column=COL_HBT,
                            value=(cand["HBT Ida"] if r == r_ida else cand["HBT Vuelta"]))
                c.border = border
                c.alignment = center

            if r_ida != fila_r1:
                c_con = ws.cell(row=r_ida, column=COL_CONEXION,
                                 value=(f'=TEXT(TIMEVALUE(H{r_ida})-TIMEVALUE(I{r_ida - 1})'
                                        f'+(TIMEVALUE(H{r_ida})<TIMEVALUE(I{r_ida - 1})),"[h]:mm")'))
                c_con.fill = conexion_lim_fill
                c_con.font = green_bold
            ws.cell(row=r_vta, column=COL_CONEXION,
                    value=(f'=TEXT(TIMEVALUE(H{r_vta})-TIMEVALUE(I{r_ida})'
                           f'+(TIMEVALUE(H{r_vta})<TIMEVALUE(I{r_ida})),"[h]:mm")'))

            filas_bloque.append((r_ida, r_vta))
            fila = r_vta + 1

        r1 = filas_bloque[0][0]
        c_ins = ws.cell(row=r1, column=COL_INS)
        c_ins.fill = dutyid_fill
        c_ins.font = bold
        c_ins.comment = Comment("Elegir el instructor de la lista", "colab_lck_a320f_nb.ipynb")
        dv_instructor.add(c_ins)
        for idx, (r_ida, r_vta) in enumerate(filas_bloque):
            if idx > 0:
                ws.cell(row=r_ida, column=COL_INS, value=f'=IF($M${r1}="","",$M${r1})')

        c_act = ws.cell(row=r1, column=COL_DESC)
        c_act.fill = dutyid_fill
        c_act.font = bold
        c_act.comment = Comment("Tipo de actividad (elegir de la lista)", "colab_lck_a320f_nb.ipynb")
        dv.add(c_act)

        for k, (r_ida, r_vta) in enumerate(filas_bloque):
            if k > 0:
                ws.cell(row=r_ida, column=COL_DESC, value=f'=IF($O${r1}="","",$O${r1})')
            ws.cell(row=r_vta, column=COL_DESC,
                    value=(f'="LIM-"&G{r_ida}&"-LIM   LA "&E{r_ida}&" ("&LEFT(H{r_ida},5)&"-"&LEFT(I{r_ida},5)&" hrs)'
                           f'  /  LA "&E{r_vta}&" ("&LEFT(H{r_vta},5)&"-"&LEFT(I{r_vta},5)&" hrs)"'))

        r_ult = filas_bloque[-1][1]
        fila_psv = fila
        c_psv_label = ws.cell(row=fila_psv, column=10, value="PSV total:")
        c_psv_label.font = bold
        c_psv_label.fill = psv_fill
        c_psv_val = ws.cell(row=fila_psv, column=11,
                             value=f'=TEXT(TIMEVALUE(I{r_ult})-TIMEVALUE(H{r1})+(I{r_ult}<H{r1}),"[h]:mm")')
        c_psv_val.fill = psv_fill

        for r_ida, r_vta in filas_bloque:
            todas_filas_pairing.append((r_ida, r_vta, r1))
            resumen_valores = [
                f"=B{r_ida}", f"=C{r_ida}", f"=D{r_ida}",
                f'=E{r_ida}&"/"&E{r_vta}',
                f'=F{r_ida}&"-"&G{r_ida}&"-"&F{r_ida}',
                f'=IF($M${r1}="","",$M${r1})',
                f'=IF($O${r1}="","",$O${r1})',
                f"=J{r_ida}",
            ]
            for j, val in enumerate(resumen_valores):
                c = ws.cell(row=r_ida, column=COL_RESUMEN_INI + j, value=val)
                c.border = border
                c.alignment = center

        fila = fila_psv + 2

    anchos = [10, 11, 9, 8, 7, 7, 9, 9, 9]
    for j, w in enumerate(anchos):
        ws.column_dimensions[ws.cell(row=1, column=2 + j).column_letter].width = w
    for col, w in ((COL_CONEXION, 9), (COL_HBT, 8), (COL_INS, 18), (COL_VACIA1, 3),
                   (COL_DESC, 38), (COL_VACIA2, 3)):
        ws.column_dimensions[ws.cell(row=1, column=col).column_letter].width = w
    for j, w in enumerate([10, 11, 9, 12, 16, 18, 24, 9]):
        ws.column_dimensions[ws.cell(row=1, column=COL_RESUMEN_INI + j).column_letter].width = w

    ws.freeze_panes = f"B{fila_labels + 1}"

    # --- hoja "Resumen Final" (equivalente al CUADRO FINAL del Archivo 10) ---
    wr = wb.create_sheet("Resumen Final")
    resumen_final_headers = ["TRIP", "Fecha", "DíaSEM", "Vuelo", "Ruta", "BP INS",
                              "INS", "ACTIVIDAD", "FLOTA", "Nombre INS", "CUPOS"]
    for j, h in enumerate(resumen_final_headers):
        c = wr.cell(row=1, column=1 + j, value=h)
        c.font = bold
        c.border = border

    P = "'Pairings NB'!"
    for i, (r_ida, r_vta, r1_bloque) in enumerate(todas_filas_pairing, start=2):
        ins_ref = f"{P}$M${r1_bloque}"
        act_ref = f"{P}$O${r1_bloque}"
        flota_ref = f"{P}J{r_ida}"
        valores = [
            f"={P}B{r_ida}",
            f"={P}C{r_ida}",
            f"={P}D{r_ida}",
            f'={P}E{r_ida}&"/"&{P}E{r_vta}',
            f'={P}F{r_ida}&"-"&{P}G{r_ida}&"-"&{P}F{r_ida}',
            f'=IFERROR(VLOOKUP({ins_ref},Instructores!$A:$C,2,FALSE),"")',
            f"={ins_ref}",
            f"={act_ref}",
            f"={flota_ref}",
            f'=IFERROR(VLOOKUP({ins_ref},Instructores!$A:$C,3,FALSE),"")',
            f'=IF({flota_ref}=320,4,3)',  # 2.4/2.18 del manual: A320=4 cupos, A319=3
        ]
        for j, val in enumerate(valores):
            c = wr.cell(row=i, column=1 + j, value=val)
            c.border = border

    for col, w in zip("ABCDEFGHIJK", (9, 11, 10, 12, 16, 10, 18, 26, 8, 40, 8)):
        wr.column_dimensions[col].width = w
    wr.freeze_panes = "A2"
    wr.auto_filter.ref = wr.dimensions

    wb.save(out_path)


## 8. Ejecutar todo

In [ ]:
OUT = "Pairings_NB_desde_Colab.xlsx"

df = cargar_bq_a_df(df_raw)
n_trips_crudos = df["trip"].nunique()

df = separar_instancias_trip(df)
n_instancias = df["trip"].nunique()

dia_duty_min_real = df.groupby("trip")["dia_duty"].min()  # ANTES de filtrar
df_filtrado = filtrar_mes_y_ruta(df, MES_OBJETIVO, ANIO_OBJETIVO)
validos, excluidos = armar_primeras_mitades(df_filtrado, dia_duty_min_real)

bloques = parear_candidatos(validos)
n_parejas = sum(1 for _, b in bloques if b is not None)
n_solos = sum(1 for _, b in bloques if b is None)

construir_bloques_nb(bloques, OUT, n_muestra=None)

print(f"Trip IDs crudos (BigQuery, se reutilizan): {n_trips_crudos}")
print(f"Instancias de pairing reales detectadas:   {n_instancias}")
print(f"Candidatos válidos (día 1 real, 2.9-2.14):  {len(validos)}")
print(f"Excluidos:                                  {len(excluidos)}")
print(f"Bloques armados: {len(bloques)} ({n_parejas} con 2 pairings, {n_solos} sin pareja)")
print(f"Archivo generado: {OUT}")


## 9. QA rápido antes de descargar

Recalcula en Python (no solo mira la estructura) que las reglas se cumplan, igual que en las
corridas anteriores ya validadas.

In [ ]:
# dep siempre LIM o arr siempre LIM
malas_rutas = validos[~((validos["Dep"]=="LIM") | (validos["Arr Vta"]=="LIM"))]
print("Rutas que no tocan LIM:", len(malas_rutas))

# ninguna llegada fuera de la lista válida
arr_malos = validos[(validos["Dep"]=="LIM") & (~validos["Arr"].isin(RUTAS_VALIDAS_ARR))]
print("Llegadas fuera de la lista válida:", len(arr_malos))

# ida y vuelta consistentes (la vuelta sale de donde llegó la ida)
incons = validos[validos["Dep Vta"] != validos["Arr"]]
print("Ida/vuelta inconsistentes:", len(incons))

# Pairing ID unicos, sin duplicados entre bloques
ids_en_bloques = []
for a, b in bloques:
    ids_en_bloques.append(a["Pairing ID"])
    if b is not None:
        ids_en_bloques.append(b["Pairing ID"])
import collections
c = collections.Counter(ids_en_bloques)
print("Pairing IDs que no aparecen exactamente 1 vez en los bloques:", sum(1 for v in c.values() if v != 1))
print("Total candidatos usados en bloques:", len(ids_en_bloques), "de", len(validos), "válidos")


## 10. Descargar el archivo

In [ ]:
from google.colab import files
files.download(OUT)


## 11. Estado y próximos pasos — SIN INVENTAR NADA

### Lo que este notebook automatiza (con QA hecho, no solo revisado por estructura)
- **Fase 0 (Insumos, pasos 01-05):** reemplazada por la query a BigQuery + `filtrar_mes_y_ruta`
  (mes, dep=LIM, rutas válidas según el documento Parte 4).
- **Fase 3 (Búsqueda de vuelos, pasos 15-20):** `armar_primeras_mitades` (primer vuelo del día,
  pairing partido, HBT>1h, salida>08:30) + `parear_candidatos` (segundo vuelo con conexión
  válida y PSV combinado ≤11h) + `construir_bloques_nb` (marcar tipo de actividad queda en un
  dropdown, para completar a mano — no se auto-asigna "LCK A320F" porque eso depende de la
  Fase 1/2, que no tenemos automatizada).

### Lo que falta, fase por fase (no se inventó ninguna lógica para esto)
- **Fase 1 — Demanda (pasos 06-10):** necesita el **Archivo 9** ("Programación LCK, Exp.R y
  Reentrenamiento") para saber quién requiere LCK este mes (columna PROGRAMAR=SI) y calcular
  cuántos días-instructor hacen falta. Vimos capturas reales de este archivo (Google Sheets),
  pero no está conectado por API — se necesitaría leerlo con `gspread` (misma autenticación de
  Google que ya usamos para BigQuery) o exportarlo a CSV a mano cada mes.
- **Fase 2 — Instructores (pasos 11-13):** necesita el **Archivo 10** para la vigencia IDE por
  flota (hojas "Restricciones INS" / "BG INS") y para reservar los slots en el **Rol de
  Instructores** — tampoco conectado.
- **Fase 4 — Consolidación, pasos 23-25 (grupos):** el bloqueo de instructor por REVA (columnas
  INS FINAL / INS F a considerar / Grupo del Archivo 10) también depende del Archivo 10 real.
  Ya vimos los 3 grupos reales (Fio-Sebas-Cris / Claudia-Jazmin-Javier-Jennifert /
  Gabriela-Jefferson-Karen-Patricia) pero **no armé la asignación automática de grupo por
  tripulante** porque esa lógica necesita el historial de REVA real, que no está confirmado
  en BigQuery (`presentacion_duty_date_lt` → `duty_presentation_date_at` sigue sin confirmar,
  y no hay un campo equivalente identificado para "quién dictó la última REVA a quién").
- **Fase 5 — Freeze (pasos 26-30):** no tenemos el archivo "202609 Freeze LP" conectado.

### Qué preguntar a otras personas antes de dar esto por definitivo
- **A Briggitte / Mario:** confirmar si la lista de rutas del documento Parte 4 (sin IQT, con
  TBP/TCQ) sigue vigente, o si cambió otra vez — se resolvió comparando 2 documentos, no
  confirmando en vivo con ellos.
- **A quien administre BigQuery / el dueño de `bbdd_pairing`:** si `duty_presentation_date_at`
  es realmente el campo correcto para `presentacion_duty_date_lt` (usado en el script WB, no
  en este de NB) — sigue sin confirmar.
- **Si se quiere automatizar Fase 1/2:** pedir acceso de lectura por API a los Google Sheets
  "9. Programación LCK..." y "10. Asignación Vuelos LCK..." (compartir la cuenta de servicio o
  habilitar `gspread` con la cuenta de Fernando), y confirmar si existe en BigQuery algún campo
  que reconstruya el historial de REVA por instructor -> sin eso, la Fase 2/4 no se puede
  automatizar sin inventar una fuente de datos que no existe.
